# Dia 2 — Baseline (Zero-Shot / Few-Shot) + Fine-Tuning QLoRA v1 (`Qwen2.5-3B-Instruct`)

Este notebook executa o pipeline completo e reproduzível do **Dia 2** em uma GPU gratuita **Google Colab T4 (15 GB VRAM)**:
1. **Setup** (`unsloth` + repositório `delivery-triage-llm-ptbr`)
2. **Baseline Zero-Shot** no conjunto `test` *(nunca treine antes de medir o baseline!)*
3. **Baseline Few-Shot (3 exemplos)** no mesmo conjunto `test`
4. **Treino QLoRA v1** (`r=16`, `alpha=32`, `train_on_responses_only=True`, monitorando `train_loss` vs `eval_loss`)
5. **Avaliação QLoRA v1** no mesmo conjunto `test`
6. **Tabela Comparativa Final** (`Zero-Shot vs Few-Shot vs QLoRA v1`) + Matriz de Confusão + Top 20 Erros

In [ ]:
# 0. Verificar GPU disponível (esperado: Tesla T4 15 GB)
!nvidia-smi

In [ ]:
# 1. Instalar Unsloth e dependências do projeto
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q pydantic scikit-learn matplotlib pyyaml datasets tqdm

In [ ]:
# 2. Clonar/atualizar o repositório delivery-triage-llm-ptbr
import os

REPO_URL = "https://github.com/sCesar-dev/delivery-triage-llm-ptbr.git"
REPO_DIR = "/content/delivery-triage-llm-ptbr"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd /content/delivery-triage-llm-ptbr

In [ ]:
# 3. Configurar acesso ao Hugging Face e caminho do dataset
# Se o dataset estiver no Google Drive, descomente as 2 linhas abaixo:
# from google.colab import drive
# drive.mount('/content/drive')

import os
from getpass import getpass

# Ajuste para o seu repo no HF (ex: 'sCesar-dev/delivery-triage-reviews-ptbr')
# OU pasta local com train.jsonl / validation.jsonl / test.jsonl (ex: '/content/drive/MyDrive/dataset')
DATASET_SOURCE = "sCesar-dev/delivery-triage-reviews-ptbr"
TEST_SOURCE = DATASET_SOURCE  # ou '/content/drive/MyDrive/dataset/test.jsonl'
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

# Opcional: login no Hugging Face se o dataset for privado ou para push_to_hub depois
# from huggingface_hub import login
# login(token=getpass("HF_TOKEN: "))

## Etapa 1 — Baseline: `Qwen2.5-3B-Instruct` Base (Zero-Shot e Few-Shot)

**Regra de ouro:** Medimos o modelo base **antes** de treinar, com decodificação determinística (`do_sample=False`, `seed=42`), no conjunto `test` blindado.

In [ ]:
# 4A. Rodar Baseline Zero-Shot no conjunto test
!python -m src.eval.evaluate \
    --dataset {TEST_SOURCE} \
    --split test \
    --base-model {BASE_MODEL} \
    --prompt-mode zero_shot \
    --run-name baseline_zero_shot \
    --model-label "Qwen2.5-3B Base (Zero-Shot)" \
    --results-dir results

In [ ]:
# 4B. Rodar Baseline Few-Shot (3 exemplos) no mesmo conjunto test
!python -m src.eval.evaluate \
    --dataset {TEST_SOURCE} \
    --split test \
    --base-model {BASE_MODEL} \
    --prompt-mode few_shot \
    --run-name baseline_few_shot \
    --model-label "Qwen2.5-3B Base (Few-Shot, 3 ex)" \
    --results-dir results

In [ ]:
# Inspecionar baseline consolidado (results/baseline_metrics.json)
!cat results/baseline_metrics.json

## Etapa 2 — Treino QLoRA v1 (4-bit NF4 + `train_on_responses_only`)

- Hiperparâmetros versionados em `configs/train_qlora_v1.yaml` (`r=16`, `alpha=32`, `lr=2e-4`, `cosine`, `2` épocas, batch efetivo `16`).
- Avalia `eval_loss` a cada `50` steps para detectar overfitting e carrega o melhor checkpoint ao final (`load_best_model_at_end=True`).

In [ ]:
# 5. Executar treino QLoRA v1 (~25-40 min na T4 para ~1.8k-2.5k exemplos)
!python -m src.train.train_qlora \
    --config configs/train_qlora_v1.yaml \
    --dataset {DATASET_SOURCE} \
    --train-split train \
    --val-split validation \
    --results-dir results

In [ ]:
# Visualizar curva train_loss vs eval_loss
from IPython.display import Image, display
display(Image(filename="results/qlora_v1_loss_curve.png"))

## Etapa 3 — Avaliação do Modelo Fine-Tunado (QLoRA v1) & Comparação Final

In [ ]:
# 6. Rodar o MESMO evaluate.py no adaptador QLoRA v1
!python -m src.eval.evaluate \
    --dataset {TEST_SOURCE} \
    --split test \
    --base-model {BASE_MODEL} \
    --adapter-path adapters/qlora_v1 \
    --prompt-mode zero_shot \
    --run-name qlora_v1 \
    --model-label "Qwen2.5-3B + QLoRA v1" \
    --results-dir results

In [ ]:
# 7. Exibir Tabela Comparativa Final + Matriz de Confusão + Resumo de Erros
from IPython.display import Image, Markdown, display

with open("results/benchmark_summary.md", encoding="utf-8") as f:
    display(Markdown(f.read()))

display(Image(filename="results/confusion_matrix_qlora_v1.png"))
!python -m src.eval.error_analysis --error-file results/error_cases_qlora_v1.jsonl

In [ ]:
# 8. (Opcional) Compactar resultados e adaptador para baixar ou salvar no Google Drive
!zip -r /content/day2_artifacts.zip results/ adapters/qlora_v1/
print("Artefatos salvos em /content/day2_artifacts.zip")